# WNGF: Weighted–Normalized Gould–Fernandez Brokerage Measure

This notebook is a **general-purpose, reusable implementation** of the WNGF measure described in:

> Zádor, Zhu, Smith & Gorgoni (2022), *A Weighted and Normalized Gould–Fernandez brokerage measure*, PLOS ONE. https://doi.org/10.1371/journal.pone.0274475

It works on **any** weighted, directed network, as long as you can provide:
1. Edge weights between nodes (a full weight matrix, or an edge list), and
2. A group/category label for each node (e.g. location, department, country, community).

It reproduces the exact broker condition from the paper (Eq. 1):

$$\frac{1}{Z_{qr}} + \frac{1}{Z_{rs}} < \frac{1}{Z_{qs}}$$

where node **r** is a broker between its predecessor **q** and successor **s** if the combined "cost" of reaching s from q via r is lower than the direct cost from q to s. An edge weight of 0 (i.e. no edge) is treated as $Z=0$, so $1/Z_{qs}=\infty$ and the condition is automatically satisfied whenever both legs of the q→r→s path exist.

**Important assumption:** like the original paper, this code assumes edge weights represent *flow / strength / frequency* (bigger = a stronger, "cheaper" tie), e.g. trade value, money, messages, interactions. If your weights represent *cost or distance* (bigger = weaker tie), invert them first (e.g. `weight = 1 / distance`) before building the network.

Each node's role frequencies are also **normalized** for group size.

---
### The five brokerage roles
For a broker **r** connecting predecessor **q** and successor **s**, based on group membership:

| Role | Condition |
|---|---|
| Coordinator | q, r, s all in the same group |
| Representative | q and r in the same group, s in a different group |
| Gatekeeper | r and s in the same group, q in a different group |
| Itinerant | q and s in the same group, r in a different group |
| Liaison | q, r, and s all in different groups |

---


## Setup

In [1]:
import math
from collections import Counter

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

%matplotlib inline


## Core WNGF functions


In [2]:
def build_wngf_network(weights, groups, directed=True, self_loops=False):
    """Build a networkx graph ready for WNGF, from either a square weight matrix or an edge list.

    Parameters
    ----------
    weights : pandas.DataFrame
        Either:
          - a square matrix, where weights.loc[i, j] is the weight of the edge from node i to node j (0 or NaN = no edge), OR
          - a long-format edge list with columns ['source', 'target', 'weight'].
    groups : dict or pandas.Series
        Maps each node name to a group/category label 
    directed : bool, default True
    self_loops : bool, default False

    Returns
    -------
    networkx.DiGraph (or Graph) with 'weight' edge attributes and a 'group' node attribute.
    """
    is_matrix = isinstance(weights, pd.DataFrame) and set(weights.index) == set(weights.columns)

    if is_matrix:
        nodes = list(weights.index)
        G = nx.DiGraph() if directed else nx.Graph()
        G.add_nodes_from(nodes)
        for i in nodes:
            for j in nodes:
                if i == j and not self_loops:
                    continue
                w = weights.loc[i, j]
                w = 0.0 if pd.isna(w) else float(w)
                G.add_edge(i, j, weight=w)
    else:
        edge_df = weights
        group_nodes = list(groups.keys()) if isinstance(groups, dict) else list(groups.index)
        nodes = sorted(set(edge_df['source']) | set(edge_df['target']) | set(group_nodes))
        G = nx.DiGraph() if directed else nx.Graph()
        G.add_nodes_from(nodes)
        for i in nodes:
            for j in nodes:
                if i == j and not self_loops:
                    continue
                G.add_edge(i, j, weight=0.0)
        for _, row in edge_df.iterrows():
            if row['source'] == row['target'] and not self_loops:
                continue
            G[row['source']][row['target']]['weight'] = float(row['weight'])

    group_map = dict(groups)
    missing = [n for n in G.nodes if n not in group_map]
    if missing:
        raise ValueError(
            f"No group label provided for {len(missing)} node(s), e.g. {missing[:10]}. "
            "Every node in the network needs a group label."
        )
    nx.set_node_attributes(G, {n: group_map[n] for n in G.nodes}, name='group')
    return G


In [3]:
def _is_broker(w_qr, w_rs, w_qs):
    """Paper Eq. (1): 1/Z_qr + 1/Z_rs < 1/Z_qs, with the convention Z_qs = 0 -> 1/Z_qs = infinity.
    Both legs of the q -> r -> s path must exist (w_qr > 0 and w_rs > 0) for r to broker at all."""
    if w_qr > 0 and w_rs > 0:
        if w_qs == 0:
            return True
        return (1.0 / w_qs) > (1.0 / w_qr + 1.0 / w_rs)
    return False


def _classify_role(g_q, g_r, g_s):
    """Classify the brokerage role of r given the group labels of q (predecessor), r (broker),
    and s (successor)."""
    if g_q == g_r == g_s:
        return 'coordinator'
    elif g_q == g_s and g_r != g_q:
        return 'itinerant'
    elif g_q != g_s and g_r == g_s:
        return 'gatekeeper'
    elif g_q == g_r and g_r != g_s:
        return 'representative'
    elif g_q != g_r and g_r != g_s and g_q != g_s:
        return 'liaison'
    return None


def _nCr(n, r):
    if r < 0 or r > n or n < 0:
        return 0
    return math.comb(n, r)


def _normalization_denominators(own_group, group_counts):
    """Group-size-based normalization denominators for each role, as defined in the paper.
    Generalized to work with any number of groups, of any size, with any labels."""
    n = group_counts[own_group]
    others = {g: c for g, c in group_counts.items() if g != own_group}

    co = _nCr(n - 1, 2) * 2 if n > 2 else 1
    ga_re = sum(c * (n - 1) for c in others.values()) if n > 1 else 1
    it = sum(_nCr(c, 2) * 2 for c in others.values() if c > 1)

    li = 0
    items = list(others.items())
    for g1, c1 in items:
        for g2, c2 in items:
            if g1 != g2:
                li += c1 * c2

    return {
        'coordinator': co or 1,
        'gatekeeper': ga_re or 1,
        'representative': ga_re or 1,
        'itinerant': it or 1,
        'liaison': li or 1,
    }


In [4]:
def compute_wngf(G, group_attr='group', normalize=True, return_pairs=False):
    """Compute WNGF brokerage roles for every node in G.

    Parameters
    ----------
    G : networkx.DiGraph
        As produced by build_wngf_network().
    group_attr : str, default 'group'
        Name of the node attribute holding each node's group label.
    normalize : bool, default True
        Whether to also compute the group-size-normalized scores
    return_pairs : bool, default False
        If True, also return a dict of the actual (predecessor, successor) pairs each node brokers for each role

    Returns
    -------
    pandas.DataFrame indexed by node, with '<role>_raw' counts and (if normalize=True)
    '<role>_norm' scores for each of the five roles, plus 'total_raw' / 'total_norm'.
    """
    group_map = nx.get_node_attributes(G, group_attr)
    if len(group_map) != G.number_of_nodes():
        raise ValueError(f"Every node needs a '{group_attr}' attribute; some are missing it.")
    group_counts = Counter(group_map.values())

    role_names = ['coordinator', 'itinerant', 'gatekeeper', 'representative', 'liaison']
    rows = []
    pairs_by_node = {}

    for node in G.nodes:
        own_group = group_map[node]
        counts = {r: 0 for r in role_names}
        pairs = {r: [] for r in role_names}

        for q in G.predecessors(node):
            for s in G.successors(node):
                if q == s:
                    continue
                w_qr = G[q][node]['weight']
                w_rs = G[node][s]['weight']
                w_qs = G[q][s]['weight'] if G.has_edge(q, s) else 0.0

                if _is_broker(w_qr, w_rs, w_qs):
                    role = _classify_role(group_map[q], own_group, group_map[s])
                    if role:
                        counts[role] += 1
                        pairs[role].append((q, s))

        row = {'node': node, 'group': own_group}
        for r in role_names:
            row[f'{r}_raw'] = counts[r]
        row['total_raw'] = sum(counts.values())

        if normalize:
            denom = _normalization_denominators(own_group, group_counts)
            for r in role_names:
                row[f'{r}_norm'] = counts[r] / denom[r]
            row['total_norm'] = sum(row[f'{r}_norm'] for r in role_names)

        rows.append(row)
        pairs_by_node[node] = pairs

    df = pd.DataFrame(rows).set_index('node')
    if return_pairs:
        return df, pairs_by_node
    return df


## Example with synthetic data

This shows the full workflow on a small made-up network, for confirmation purpose.

In [5]:
# Build a small example network 
np.random.seed(42)

n_nodes = 15
node_names = [f"node_{i}" for i in range(n_nodes)]

# assign each node to one of three groups
group_labels = ['Group_A', 'Group_B', 'Group_C']
groups = {node: group_labels[i % 3] for i, node in enumerate(node_names)}

# random weighted adjacency matrix
raw = np.random.choice([0, 0, 0, 1, 2, 3, 5, 8, 13], size=(n_nodes, n_nodes))
np.fill_diagonal(raw, 0)
weight_matrix = pd.DataFrame(raw, index=node_names, columns=node_names).astype(float)

weight_matrix.head()


,node_0,node_1,node_2,node_3,node_4,node_5,node_6,node_7,node_8,node_9,node_10,node_11,node_12,node_13,node_14
node_0,0.0,1.0,8.0,2.0,5.0,0.0,5.0,8.0,2.0,1.0,8.0,8.0,0.0,3.0,2.0
node_1,0.0,0.0,3.0,0.0,2.0,0.0,3.0,13.0,0.0,0.0,5.0,1.0,13.0,0.0,2.0
node_2,0.0,5.0,0.0,13.0,5.0,0.0,1.0,13.0,0.0,13.0,2.0,0.0,1.0,5.0,8.0
node_3,0.0,0.0,1.0,0.0,8.0,1.0,0.0,3.0,3.0,1.0,3.0,0.0,0.0,1.0,8.0
node_4,5.0,13.0,8.0,2.0,0.0,2.0,8.0,13.0,13.0,0.0,13.0,5.0,13.0,8.0,0.0


In [6]:
G_example = build_wngf_network(weight_matrix, groups)
results_example = compute_wngf(G_example)

results_example.sort_values('total_norm', ascending=False).head(10)


,group,coordinator_raw,itinerant_raw,gatekeeper_raw,representative_raw,liaison_raw,total_raw,coordinator_norm,itinerant_norm,gatekeeper_norm,representative_norm,liaison_norm,total_norm
node,,,,,,,,,,,,,
node_4,Group_B,5,17,19,18,25,84,0.416667,0.425,0.475,0.450,0.50,2.266667
node_7,Group_B,2,18,9,11,15,55,0.166667,0.450,0.225,0.275,0.30,1.416667
node_11,Group_C,2,8,15,7,8,40,0.166667,0.200,0.375,0.175,0.16,1.076667
node_2,Group_C,1,12,7,5,17,42,0.083333,0.300,0.175,0.125,0.34,1.023333
node_14,Group_C,2,9,13,2,10,36,0.166667,0.225,0.325,0.050,0.20,0.966667
node_0,Group_A,3,6,7,10,7,33,0.250000,0.150,0.175,0.250,0.14,0.965000
node_3,Group_A,2,6,4,17,5,34,0.166667,0.150,0.100,0.425,0.10,0.941667
node_8,Group_C,2,6,10,8,7,33,0.166667,0.150,0.250,0.200,0.14,0.906667
node_13,Group_B,2,10,8,3,10,33,0.166667,0.250,0.200,0.075,0.20,0.891667


## Using your own data

You have two easy ways to bring in your own network. Adapt whichever works for your data.


In [ ]:
# Either a square weight matrix + a separate group/attribute file 
#
# weight_matrix = pd.read_csv('my_weight_matrix.csv', index_col=0)
# attributes = pd.read_csv('my_node_attributes.csv', index_col=0)
# groups = attributes['group_column_name'].to_dict()
#
# G = build_wngf_network(weight_matrix, groups)
# results = compute_wngf(G)


In [ ]:
# Or a long-format edge list (source, target, weight) + a group mapping
#
# edge_list = pd.read_csv('my_edges.csv')  # must have columns: source, target, weight
# attributes = pd.read_csv('my_node_attributes.csv', index_col=0)
# groups = attributes['group_column_name'].to_dict()
#
# G = build_wngf_network(edge_list, groups)
# results = compute_wngf(G)
